# Fase 3 · Streaming y gobernanza NYC311
**Dante Aguirre · Magíster en Data Science UDD · 1 de octubre de 2026**

Este notebook consolida las fases 1 y 2 y verifica la extensión Kafka–Structured Streaming con evidencia ejecutada. La pregunta sigue siendo qué solicitudes presentan riesgo temprano de atención prolongada respecto de su tipo de reclamo. El p75 histórico por tipo define la etiqueta; **no es un SLA oficial**.

La solución técnica pasa los controles del replay, pero el modelo aún requiere revisión operativa: selecciona el 79,27% de la prueba y presenta diferencias territoriales relevantes. Se recomienda un piloto supervisado, sin asignación automática de atención.

**Alcance:** 13.072.334 registros auditados; entrenamiento final con 9.189.270 y prueba con 3.365.506 casos etiquetables. Streaming: todo el 1 de enero de 2024 (7.269 solicitudes), más un evento real del día siguiente para cerrar ventanas. No se simuló el flujo de los cuatro años.

La ejecución por defecto lee y contrasta las evidencias adjuntas, sin volver a entrenar ni arrancar Kafka. Para repetir el procesamiento real, ver la última sección y `README.md`.

In [ ]:
from pathlib import Path
import json, hashlib, sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
assert (ROOT / "evidence/final_result.json").exists(), "Abrir desde la carpeta del notebook"
E = ROOT / "evidence"
S = ROOT / "stream_run/evidence"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
def read_json(path): return json.loads(path.read_text())
model = read_json(E / "final_result.json")
stream = read_json(S / "summary.json")
pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
display(pd.Series(read_json(E / "environment.json")))

## 1. Continuidad y evaluación batch
Bronze conserva los CSV y sus hashes; Silver tipa fechas y categorías y particiona por año; Gold utiliza información disponible al ingreso. El cierre, estado y descripción de resolución se excluyen de los predictores. Los umbrales p75 se calculan con historia cerrada anterior a cada corte y al menos 100 casos por tipo; los tipos no soportados reciben abstención.

Se evaluaron 18 ajustes (9 candidatos × 2 cortes temporales) sobre una muestra de desarrollo del 0,2%, con umbrales calculados en la historia completa. El GBT final se ajustó al 100% elegible; no se reentrena en fase 3. Un modelo provisional del 5% ya había observado la prueba: el reintento completo mantuvo la selección previa, sin retuning con 2024. Se conserva esta limitación del protocolo.

America/New_York corrige las duraciones negativas a 25.428 frente a 26.113 con America/Santiago. Se excluyen de la etiqueta y se mantiene la conciliación. La población de prueba excluye solicitudes abiertas o inválidas y tipos sin historia suficiente; no representa todos los ingresos.

In [ ]:
m = model["metrics"]
assert m["tp"] + m["fp"] + m["fn"] + m["tn"] == m["n"]
assert m["tp"] + m["fn"] == m["positives"]
assert model["decision"]["test_used_for_selection"] is False
display(pd.Series({k:m[k] for k in ["train_n","n","areaUnderPR","areaUnderROC","precision","recall","f1","alert_fraction","fit_seconds","infer_seconds"]}))
experiments = pd.read_csv(E / "experiment_summary.csv")
display(experiments.sort_values("pr_mean",ascending=False))

## 2. Benchmarks sobre consultas reales
Se incorpora el comentario docente: medir `monthly_borough` y `agency_window` para apoyar la decisión del proceso completo. Las mediciones usan toda Silver, calentamiento y tres repeticiones con resultado materializado, sin persistencia Spark; la caché del sistema operativo no está controlada. Se comparan **local[1] y local[2] de Spark**, no nuevos resultados intermotor.

Los conteos y rankings coinciden; los percentiles aproximados presentan dos medianas y cuatro diferencias derivadas de hasta 0,1 horas. La fase 1 favorecía DuckDB en consultas locales simples; sus series de PDF y notebook no coinciden, por lo que aquí no se reutilizan como una comparación homogénea de las consultas reales. No se afirma que Spark sea el motor local más rápido.

In [ ]:
bench = pd.read_csv(E / "batch_query_benchmark.csv")
display(bench)
# Mediana del tiempo de cada par en la evidencia histórica, no suma de medianas aisladas.
pair_one, pair_two = 18.905376, 12.034387
print(f"Razón de medianas del par local[1]/local[2]: {pair_one/pair_two:.3f}")
stages = pd.Series({"Preparación":253.690565458,"Consultas reales local[2]":pair_two,
                    "Ajuste final":m["fit_seconds"],"Inferencia":m["infer_seconds"]})
display(stages.rename("segundos"))
print(f"Suma orientativa: {stages.sum()/60:.2f} min; ajuste: {100*m['fit_seconds']/stages.sum():.1f}%")

La suma de 39,87 minutos reúne etapas de corridas distintas: **no es una medición de pared de extremo a extremo**. Excluye tuning, arranques, evaluación y exportación. El ajuste domina esta suma (88,5%); optimizar solo consultas no resuelve ese costo. Mantener Spark permite preparación, MLlib y streaming en un mismo entorno, mientras DuckDB sigue siendo una alternativa de exploración local. El ensayo con dos núcleos no demuestra escalamiento entre máquinas.

## 3. Contrato y flujo Kafka
`Silver → JSONL mínimo → Kafka (3 particiones) → Spark → watermark 2 h → ventanas 1 h × borough → Parquet append + checkpoint`.

El contrato contiene identificador público, timestamp con zona, borough y campos de auditoría de emisión. No publica direcciones, coordenadas ni texto libre en Kafka. El identificador sigue siendo vinculable a la fuente pública, por lo que no se declara anonimización irreversible. El broker de demostración solo escucha localhost, con una réplica y retención de 24 horas; no tiene alta disponibilidad ni seguridad de producción.

Se conserva el timestamp real y se retrasa exclusivamente la emisión de dos solicitudes: una de las 06 h se envía a las 07 h; una de las 00 h a las 08 h. Un evento real del día siguiente a partir de las 03 h avanza el watermark. El tiempo del evento, no el reloj de pared, determina el cierre. La demostración controlada no estima la distribución real de atrasos.

In [ ]:
meta = read_json(E / "replay_input.json")
raw = ROOT / "data/replay_events.jsonl"
assert hashlib.sha256(raw.read_bytes()).hexdigest() == meta["sha256"]
events = pd.read_json(raw,lines=True,dtype={"event_id":str})
assert events.event_id.is_unique
assert len(events)==7270 and (events.delay_kind!="frontier").sum()==7269
assert meta["timestamps_changed"] is False and meta["sampled_within_day"] is False
display(events[events.delay_kind!="on_time"])
display(pd.Series(stream))

## 4. Validación de ventanas y recuperación
La consulta real consumió 7.270 mensajes. Retuvo 7.268 solicitudes del día, aceptó el atraso leve y descartó una fila extrema. Se verificaron 125 ventanas por territorio contra un oráculo batch independiente. La solicitud de frontera queda en estado y no integra los totales del día.

Al detener y reanudar la consulta con el mismo checkpoint, las 52 filas ya escritas permanecieron en 52. Esto verifica continuidad del ensayo; **no implica deduplicar eventos republicados con offsets nuevos**. El checkpoint archivado no se reutiliza con un broker distinto. Cada reproducción crea un tópico y salida nuevos.

In [ ]:
windows = pd.read_csv(S / "windows.csv")
# Oráculo reconstruido aquí desde los mensajes originales, sin usar el resultado Spark.
expected = events[~events.delay_kind.isin(["extreme","frontier"])].copy()
expected["window_start"] = pd.to_datetime(expected.event_ts,utc=True).dt.tz_convert("America/New_York").dt.floor("h").dt.tz_localize(None)
oracle = expected.groupby(["window_start","borough"]).size().rename("requests").reset_index()
observed = windows.copy(); observed["window_start"] = pd.to_datetime(observed.window_start)
pd.testing.assert_frame_equal(oracle.sort_values(["window_start","borough"]).reset_index(drop=True),
 observed[["window_start","borough","requests"]].sort_values(["window_start","borough"]).reset_index(drop=True),check_dtype=False)
assert windows.requests.sum()==7268 and windows.mild.sum()==1 and windows.extreme.sum()==0
assert stream["late_rows_dropped"]==1 and stream["restart"]["duplicate_rows_on_restart"]==0
print("Controles de conteos, atraso y reinicio: correctos.")
progress=read_json(S / "progress.json")
rows=[]
for p in progress:
 for st in p.get("stateOperators",[]):
  rows.append({"batch":p["batchId"],"filas_estado":st["numRowsTotal"],"descartadas":st["numRowsDroppedByWatermark"]})
state=pd.DataFrame(rows).drop_duplicates("batch").sort_values("batch")
fig,ax=plt.subplots(figsize=(8,3.2));ax.plot(state.batch,state.filas_estado,marker="o",ms=3,color="#176B87")
ax.set(xlabel="Microbatch",ylabel="Filas en estado",title="El watermark mantiene acotado el estado en el replay")
fig.tight_layout();fig.savefig(FIG/"estado_streaming.png",dpi=180);plt.show()

## 5. Rendimiento y alertas de volumen
El replay tardó 77,63 s, más 2,46 s de arranque Spark. La tasa efectiva de 93,64 mensajes/s está regulada por el productor que espera a cada microbatch: **no es una prueba de saturación**. El máximo muestreado de RSS Python + Spark fue 733,17 MiB; excluye el broker Kafka independiente y no es memoria física exclusiva. El estado llegó a 22 filas y terminó con una.

Las alertas de volumen se calculan después del cierre sobre Parquet: solicitudes > 2 × línea base y al menos 20. La línea base es el promedio de tasas horarias mensuales de 2023 por territorio, sin estacionalidad intradía. No es el clasificador de demora ni un sink continuo de alertas. En este día hubo **cero alertas**, resultado que se conserva.

In [ ]:
baseline=pd.read_csv(ROOT/"data/baseline_2023.csv")
alerts=windows.merge(baseline,on="borough",how="left",validate="many_to_one")
alerts["alert"]=(alerts.requests>2*alerts.baseline_hourly)&(alerts.requests>=20)&alerts.baseline_hourly.notna()
assert int(alerts.alert.sum())==stream["volume_alert_windows"]==0
# Comprobación separada de la regla, con valores artificiales declarados; no añade eventos al replay.
rule=lambda n,b: n>2*b and n>=20
assert not rule(20,10) and rule(21,10) and not rule(19,5)
display(alerts.groupby("borough").agg(ventanas=("requests","size"),solicitudes=("requests","sum"),alertas=("alert","sum")))
print("Memoria muestreada MiB:",stream["max_python_spark_rss_bytes"]/1024**2)

## 6. Equidad del modelo de fase 2
Se usan los conteos completos de la prueba del modelo final, sin volver a ajustar el umbral 0,2. La selección para revisión humana se interpreta como beneficio **hipotético**, condicionado a capacidad real. Referencia: grupo de mayor tamaño dentro de cada dimensión (Brooklyn y ONLINE), elegida sin buscar la mayor brecha.

SPD = tasa de selección del grupo − tasa de referencia; DI = tasa del grupo / tasa de referencia. TPR = TP / positivos; FPR = FP / negativos. Se comparan también brechas de prevalencia: diferencia entre magnitudes de SPD y de brecha de etiqueta, con tolerancia descriptiva de 2 puntos porcentuales. No es una prueba causal. DI < 0,8 es una señal diagnóstica, no un dictamen legal.

Borough y canal son dimensiones operativas, **no atributos protegidos**. Faltan atributos protegidos y cruces individuales para una auditoría demográfica/interseccional. Se excluye de interpretación OTHER, con 21 casos; los desconocidos se presentan aparte.

In [ ]:
g=pd.read_csv(E/"test_subgroups.csv")
parts=[]
for dim in ["borough","open_data_channel_type"]:
 t=g[g.group_variable.eq(dim)].copy()
 assert t.n.sum()==m["n"] and t.tp.sum()==m["tp"]
 assert t.alerts.sum()==m["tp"]+m["fp"] and t.positives.sum()==m["positives"]
 t["selection_rate"]=t.alerts/t.n;t["prevalence"]=t.positives/t.n
 t["tpr"]=t.tp/t.positives;t["fpr"]=(t.alerts-t.tp)/(t.n-t.positives)
 ref=t.sort_values(["n","group_value"],ascending=[False,True]).iloc[0]
 t["SPD"]=t.selection_rate-ref.selection_rate;t["DI"]=t.selection_rate/ref.selection_rate
 t["TPR_gap"]=t.tpr-ref.tpr;t["label_gap"]=t.prevalence-ref.prevalence
 t["reference"]=ref.group_value;parts.append(t)
fair=pd.concat(parts,ignore_index=True)
saved=pd.read_csv(E/"fairness.csv")
for col in ["SPD","DI","TPR_gap","label_gap"]: np.testing.assert_allclose(fair[col],saved[col],atol=1e-12)
display(fair.loc[fair.n>=100,["group_variable","group_value","n","reference","selection_rate","SPD","DI","tpr","TPR_gap"]])

In [ ]:
b=fair[(fair.group_variable=="borough") & (fair.group_value.isin(["BRONX","BROOKLYN","MANHATTAN","QUEENS","STATEN ISLAND"]))].sort_values("selection_rate")
y=np.arange(len(b));fig,ax=plt.subplots(figsize=(8,3.4))
ax.barh(y-.17,b.selection_rate*100,.32,label="Selección para revisión",color="#176B87")
ax.barh(y+.17,b.prevalence*100,.32,label="Demora prolongada observada",color="#B4C9CD")
ax.set(yticks=y,yticklabels=b.group_value,xlim=(0,105),xlabel="Porcentaje de solicitudes",title="La selección difiere más que la prevalencia entre territorios")
ax.legend(loc="lower right",fontsize=8);fig.tight_layout();fig.savefig(FIG/"fairness_borough.png",dpi=180);plt.show()

Staten Island tiene selección 34,34% frente a 70,43% en Brooklyn (SPD −36,09 pp; DI 0,488), y TPR 41,29% frente a 70,52%. Queens y Bronx se seleccionan casi por completo. La mezcla de tipos, umbrales, canales y capacidad histórica puede influir; las brechas no prueban discriminación causal. Se recomienda estudiar errores por tipo y territorio con datos de desarrollo y validación futura, sin elegir mitigaciones con el test 2024. Con 79,27% de alertas y precisión 30,33%, tampoco se demuestra un beneficio de priorización en producción.

## 7. FinOps mensual
Precios públicos consultados el 1 de octubre de 2026: Lightsail Linux con IPv4 pública, 8 GB/2 vCPU USD 44/mes; 16 GB/4 vCPU USD 84/mes. Snapshot retenido USD 0,05 por GB-mes. Referencias en `FUENTES.md`; región de planificación us-east-1, disponibilidad y cuotas por verificar antes de contratar.

Supuestos: una VM continua, Kafka y Spark juntos, sin alta disponibilidad; 50 GB retenidos en snapshots, no 50 GB nuevos diarios; transferencia dentro de la cuota y logs dentro del disco incluido. Añadir 20% de contingencia y 2 h/mes a USD 20/h como supuesto de trabajo humano. Excluye impuestos, sobreconsumos, observabilidad administrada y operación de cumplimiento. No se compraron recursos cloud.

In [ ]:
cost=pd.read_csv(E/"finops_scenarios.csv")
np.testing.assert_allclose(cost.infra_usd_month,cost.vm_usd_month+cost.snapshot_stored_gb*.05)
np.testing.assert_allclose(cost.total_with_ops_usd_month,cost.infra_usd_month*1.2+40)
display(cost[["scenario","snapshot_stored_gb","infra_usd_month","budget_infra_usd_month","total_with_ops_usd_month"]])
base=cost[cost.snapshot_stored_gb==50]
fig,ax=plt.subplots(figsize=(8,3.2));left=np.zeros(2)
for col,label,color in [("vm_usd_month","VM","#176B87"),("snapshot_usd_month","Snapshot","#A7C6CD"),("contingency_20pct","Contingencia","#D7B778")]:
 ax.barh(base.scenario,base[col],left=left,label=label,color=color);left+=base[col].to_numpy()
ax.set(xlabel="USD por mes",title="Presupuesto de infraestructura con 50 GB de snapshots")
ax.legend(loc="lower right",fontsize=8);fig.tight_layout();fig.savefig(FIG/"finops.png",dpi=180);plt.show()

El piloto de 8 GB cuesta USD 46,50/mes de infraestructura; USD 55,80 con contingencia y USD 95,80 incluyendo el trabajo supuesto. Para 16 GB: USD 86,50, USD 103,80 y USD 143,80 respectivamente. No se prorratea una VM aprovisionada por los minutos activos de Spark.

Se recomienda comenzar con un piloto de 8 GB y separar horarios de entrenamiento y streaming. Escalar a 16 GB solo ante presión persistente de memoria o atraso medido; investigar primero particiones, volumen y consultas. El Mac no valida la capacidad de esa VM. Kafka, almacenamiento y servicios de fondo también consumen memoria. Este replay no justifica un clúster distribuido ni garantiza una tasa máxima.

## 8. Calidad, gobernanza e IA generativa
**Implementado:** snapshots y hashes, splits temporales, exclusión de predictores posteriores al ingreso, tracking MLflow, versión de modelo y encoder, conciliaciones completas, contrato Kafka mínimo, evidencia de offsets/checkpoint y brechas reproducibles. Los archivos fuente conservan el historial, y los resultados de fase 2 no fueron retocados para mejorar la equidad.

**Antes de operar:** designar responsable y finalidad, verificar base jurídica y condiciones de NYC Open Data, controles de acceso y cifrado, procedimiento de derechos y reclamos, monitoreo de calidad/deriva/brechas, política de retención y eliminación, continuidad y evaluación de impacto. La retención Kafka de 24 h está configurada; la eliminación programada de copias académicas y snapshots no está implementada. Un identificador público puede vincularse con la fuente; no se garantiza anonimato.

Al 1 de octubre de 2026, la Ley 21.719 tiene vigencia diferida al **1 de diciembre de 2026**; la revisión distingue la normativa actualmente aplicable de los controles de preparación. GDPR requiere evaluar su ámbito territorial y material; no se presume aplicable automáticamente por usar NYC311. Minimización, seguridad y revisión humana se adoptan como criterios de diseño, sin certificar cumplimiento. Una revisión jurídica institucional debe preceder una operación real. Fuentes oficiales y fechas en `FUENTES.md`.

**Declaración de IA:** se utilizó Codex/OpenAI para inspeccionar la pauta y antecedentes, proponer y escribir código, ejecutar controles, preparar análisis y redactar notebook e informe. Prompt literal de fase 3: «ahora tenemos que hacer la fase 3, según lo que ya hicimos de fase 1 y fase 2, tienes el detalle en el pdf sobre proyecto integrador». No se inventa una transcripción exhaustiva de los prompts previos. La validación computacional incluye Kafka real, oráculo independiente, hashes, sumas de confusión, recomputación de fairness y costos, ejecución del notebook y revisión de páginas. La revisión final académica corresponde al autor; no se declara una revisión humana que no haya ocurrido.

## 9. Reproducción y entregables
`README.md` detalla instalación, preparación de crudos, tuning, benchmark, entrenamiento final y exportación dentro de `phase2`, seguida de reconstrucción del JSONL y ejecución Kafka. Los CSV originales se identifican por hash y no se duplican en el ZIP; el replay mínimo sí está incluido. Se requiere Java 17, un entorno Python compatible y carpetas locales fuera de iCloud. El modelo, encoder y MLflow originales están incluidos.

La siguiente celda está desactivada por defecto. Para activar, establecer `RUN_STREAM=True`, definir `NYC311_RUNTIME` con la ruta del runtime preparado mediante `setup_kafka.py` y asegurar broker en localhost:19092. Produce una salida nueva, sin sustituir la evidencia histórica del informe. Para analizar esa nueva salida, cambiar `S` al nuevo directorio y revisar cualquier diferencia. Los tiempos dependen del equipo.

No se ha vuelto a ejecutar el entrenamiento completo en fase 3. La reproducción de la demostración y la lectura de evidencia son rutas diferentes de la reconstrucción completa desde crudos.

In [ ]:
RUN_STREAM = False
if RUN_STREAM:
 import os, datetime
 runtime=Path(os.environ["NYC311_RUNTIME"])
 name="stream_run_"+datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
 subprocess.run([sys.executable,str(ROOT/"streaming.py"),"--root",str(ROOT),"--jars",str(runtime/"jars"),"--run-name",name],check=True)
 print("Nueva evidencia:",ROOT/name/"evidence")
else:
 print("Modo evidencia completado. No se reentrenó ni se inició otra consulta Kafka.")

## Fuentes y trazabilidad
- Pauta Proyecto Integrador ADBD 2026, clases 6–7 y Lab7; copias textuales de referencia en `evidence`.
- Fases 1 y 2 locales, modelo final y experimento de revisión de consultas. Archivos completos de fase 2 en `phase2`.
- [Apache Spark Structured Streaming](https://spark.apache.org/docs/3.5.6/structured-streaming-programming-guide.html): referencia de semántica; ejecución fijada en 3.5.4.
- [AWS Lightsail bundles](https://docs.aws.amazon.com/lightsail/latest/userguide/amazon-lightsail-bundles.html) y [precios](https://aws.amazon.com/lightsail/pricing/).
- [BCN Ley 21.719](https://bcn.cl/gJo3hf) y [GDPR EUR-Lex](https://eur-lex.europa.eu/eli/reg/2016/679/oj).

Fecha de consulta: 1 de octubre de 2026. Ver `FUENTES.md`, manifiesto de archivos y `evidence` para la pista de auditoría. Las extensiones no probadas se identifican como trabajo futuro, no como resultados.